In [4]:
import os
import numpy as np
import mne
import matplotlib.pyplot as plt

from mne.decoding import CSP
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

In [5]:
data_folder = "../data/BCI_IV_2a"

train_file = os.path.join(data_folder, "A01T.gdf")
test_file = os.path.join(data_folder, "A01E.gdf")

print("Training file:", train_file)
print("Testing file:", test_file)

Training file: ../data/BCI_IV_2a\A01T.gdf
Testing file: ../data/BCI_IV_2a\A01E.gdf


In [6]:
print("Loading A01T...")

raw_train = mne.io.read_raw_gdf(
    train_file,
    preload=True
)

print(raw_train)
print("Channels:", len(raw_train.ch_names))
print("Sampling frequency:", raw_train.info["sfreq"])

Loading A01T...
Extracting GDF parameters from ../data/BCI_IV_2a\A01T.gdf...
Setting channel info structure...
Could not determine channel type of the following channels, they will be set as EEG:
EEG-Fz, EEG, EEG, EEG, EEG, EEG, EEG, EEG-C3, EEG, EEG-Cz, EEG, EEG-C4, EEG, EEG, EEG, EEG, EEG, EEG, EEG, EEG-Pz, EEG, EEG, EOG-left, EOG-central, EOG-right
Creating raw.info structure...
Reading 0 ... 672527  =      0.000 ...  2690.108 secs...


C:\Users\chahi\AppData\Local\Python\pythoncore-3.11-64\Lib\contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


<RawGDF | A01T.gdf, 25 x 672528 (2690.1 s), ~128.3 MiB, data loaded>
Channels: 25
Sampling frequency: 250.0


In [7]:
print("Loading A01E...")

raw_test = mne.io.read_raw_gdf(
    test_file,
    preload=True
)

print(raw_test)
print("Channels:", len(raw_test.ch_names))
print("Sampling frequency:", raw_test.info["sfreq"])

Loading A01E...
Extracting GDF parameters from ../data/BCI_IV_2a\A01E.gdf...
Setting channel info structure...
Could not determine channel type of the following channels, they will be set as EEG:
EEG-Fz, EEG, EEG, EEG, EEG, EEG, EEG, EEG-C3, EEG, EEG-Cz, EEG, EEG-C4, EEG, EEG, EEG, EEG, EEG, EEG, EEG, EEG-Pz, EEG, EEG, EOG-left, EOG-central, EOG-right
Creating raw.info structure...
Reading 0 ... 686999  =      0.000 ...  2747.996 secs...


C:\Users\chahi\AppData\Local\Python\pythoncore-3.11-64\Lib\contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


<RawGDF | A01E.gdf, 25 x 687000 (2748.0 s), ~131.1 MiB, data loaded>
Channels: 25
Sampling frequency: 250.0


In [8]:
events_train, event_dict_train = mne.events_from_annotations(raw_train)
events_test, event_dict_test = mne.events_from_annotations(raw_test)

print("A01T events:")
print(event_dict_train)

print("\nA01E events:")
print(event_dict_test)

Used Annotations descriptions: [np.str_('1023'), np.str_('1072'), np.str_('276'), np.str_('277'), np.str_('32766'), np.str_('768'), np.str_('769'), np.str_('770'), np.str_('771'), np.str_('772')]
Used Annotations descriptions: [np.str_('1023'), np.str_('1072'), np.str_('276'), np.str_('277'), np.str_('32766'), np.str_('768'), np.str_('783')]
A01T events:
{np.str_('1023'): 1, np.str_('1072'): 2, np.str_('276'): 3, np.str_('277'): 4, np.str_('32766'): 5, np.str_('768'): 6, np.str_('769'): 7, np.str_('770'): 8, np.str_('771'): 9, np.str_('772'): 10}

A01E events:
{np.str_('1023'): 1, np.str_('1072'): 2, np.str_('276'): 3, np.str_('277'): 4, np.str_('32766'): 5, np.str_('768'): 6, np.str_('783'): 7}


In [9]:
event_id = {
    "LEFT": 7,
    "RIGHT": 8,
    "FOOT": 9,
    "TONGUE": 10
}

In [10]:
print("A01T target events:")

for name, code in event_id.items():
    count = np.sum(events_train[:, 2] == code)
    print(name, code, ":", count)

print("\nA01E target events:")

for name, code in event_id.items():
    count = np.sum(events_test[:, 2] == code)
    print(name, code, ":", count)

A01T target events:
LEFT 7 : 72
RIGHT 8 : 72
FOOT 9 : 72
TONGUE 10 : 72

A01E target events:
LEFT 7 : 288
RIGHT 8 : 0
FOOT 9 : 0
TONGUE 10 : 0


In [11]:
print("A01E event counts:")

unique, counts = np.unique(events_test[:, 2], return_counts=True)

for event, count in zip(unique, counts):
    print(event, count)

A01E event counts:
1 7
2 1
3 1
4 1
5 9
6 288
7 288


In [12]:
print("\nA01E annotation descriptions:")
print(raw_test.annotations.description)


A01E annotation descriptions:
['32766' '276' '32766' '277' '32766' '1072' '32766' '768' '783' '768'
 '783' '768' '783' '768' '783' '768' '783' '768' '783' '768' '783' '768'
 '783' '768' '783' '768' '783' '768' '783' '768' '783' '768' '783' '768'
 '783' '768' '783' '768' '783' '768' '783' '768' '783' '768' '1023' '783'
 '768' '1023' '783' '768' '783' '768' '783' '768' '783' '768' '783' '768'
 '783' '768' '783' '768' '783' '768' '783' '768' '783' '768' '783' '768'
 '783' '768' '783' '768' '783' '768' '1023' '783' '768' '783' '768' '783'
 '768' '783' '768' '783' '768' '783' '768' '783' '768' '783' '768' '783'
 '768' '783' '768' '783' '768' '783' '768' '783' '768' '783' '768' '783'
 '32766' '768' '783' '768' '783' '768' '783' '768' '783' '768' '783' '768'
 '783' '768' '1023' '783' '768' '783' '768' '783' '768' '783' '768' '783'
 '768' '783' '768' '783' '768' '783' '768' '783' '768' '783' '768' '783'
 '768' '783' '768' '783' '768' '783' '768' '1023' '783' '768' '783' '768'
 '783' '768' '78

In [13]:
print("Files in dataset folder:\n")

for file in sorted(os.listdir(data_folder)):
    print(file)

Files in dataset folder:

A01E.gdf
A01T.gdf
A02E.gdf
A02T.gdf
A03E.gdf
A03T.gdf
A04E.gdf
A04T.gdf
A05E.gdf
A05T.gdf
A06E.gdf
A06T.gdf
A07E.gdf
A07T.gdf
A08E.gdf
A08T.gdf
A09E.gdf
A09T.gdf


In [15]:
from scipy.io import loadmat

label_file = "../data/true_labels/A01E.mat"

labels_data = loadmat(label_file)

print(labels_data.keys())

dict_keys(['__header__', '__version__', '__globals__', 'classlabel'])


In [16]:
y_test = labels_data["classlabel"].reshape(-1)

print("Number of labels:", len(y_test))
print("Labels:", y_test)
print("Class counts:", np.unique(y_test, return_counts=True))

Number of labels: 288
Labels: [1 2 2 1 2 1 2 3 2 4 1 3 2 1 4 4 4 4 4 1 3 2 1 1 3 4 1 3 3 3 1 2 1 2 2 1 2
 3 2 3 3 4 3 3 4 4 4 4 4 3 2 1 1 2 3 4 2 3 1 1 1 4 2 2 1 1 3 1 2 4 4 3 1 4
 4 2 4 4 2 1 2 3 3 3 4 3 1 4 2 3 2 3 4 2 3 1 1 1 4 2 1 3 1 3 2 4 1 3 3 1 3
 2 4 4 4 3 1 4 2 4 2 1 3 2 1 3 3 1 3 4 4 2 1 2 4 2 4 3 2 2 2 3 4 1 2 4 1 3
 3 4 1 1 3 2 4 4 4 2 1 3 2 4 1 4 3 2 4 4 1 2 2 3 4 2 1 1 4 2 1 3 2 2 3 1 4
 3 3 3 3 1 2 1 2 1 1 3 3 2 3 4 1 4 1 1 2 4 3 2 4 3 4 3 4 2 2 4 1 2 2 2 3 4
 1 4 1 3 1 4 1 3 1 2 3 3 4 1 2 4 2 3 3 1 4 2 4 1 1 3 3 2 4 2 2 1 2 4 4 2 2
 2 2 4 4 3 4 1 2 3 2 1 4 1 4 1 1 1 1 3 3 4 2 3 3 3 4 3 1 3]
Class counts: (array([1, 2, 3, 4], dtype=uint8), array([72, 72, 72, 72]))


In [25]:
# ============================
# PREPARE A01T
# ============================

# Keep EEG channels only
raw_train.pick_types(eeg=True)

# Get events
events_train, event_dict_train = mne.events_from_annotations(raw_train)

# Motor imagery classes in A01T
event_id_train = {
    "LEFT": 7,
    "RIGHT": 8,
    "FOOT": 9,
    "TONGUE": 10
}

# Create epochs
epochs_train = mne.Epochs(
    raw_train,
    events_train,
    event_id=event_id_train,
    tmin=0,
    tmax=4,
    baseline=None,
    preload=True,
    verbose=False
)

X_train = epochs_train.get_data()

# Convert MNE event codes to the official class labels
# 7  -> 1 = Left Hand
# 8  -> 2 = Right Hand
# 9  -> 3 = Foot
# 10 -> 4 = Tongue

label_mapping = {
    7: 1,
    8: 2,
    9: 3,
    10: 4
}

y_train = np.array([
    label_mapping[label]
    for label in epochs_train.events[:, -1]
])

print("A01T X:", X_train.shape)
print("A01T y:", y_train.shape)
print("A01T classes:", np.unique(y_train, return_counts=True))

print("A01T X:", X_train.shape)
print("A01T y:", y_train.shape)

NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Used Annotations descriptions: [np.str_('1023'), np.str_('1072'), np.str_('276'), np.str_('277'), np.str_('32766'), np.str_('768'), np.str_('769'), np.str_('770'), np.str_('771'), np.str_('772')]
A01T X: (288, 25, 1001)
A01T y: (288,)
A01T classes: (array([1, 2, 3, 4]), array([72, 72, 72, 72]))
A01T X: (288, 25, 1001)
A01T y: (288,)


In [26]:
# ============================
# PREPARE A01E
# ============================

# Keep EEG channels only
raw_test.pick_types(eeg=True)

# Get events
events_test, event_dict_test = mne.events_from_annotations(raw_test)

# 783 marks the beginning of each evaluation trial
event_id_test = {
    "TRIAL": 7
}

epochs_test = mne.Epochs(
    raw_test,
    events_test,
    event_id=event_id_test,
    tmin=0,
    tmax=4,
    baseline=None,
    preload=True,
    verbose=False
)

X_test = epochs_test.get_data()

print("A01E X:", X_test.shape)

NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Used Annotations descriptions: [np.str_('1023'), np.str_('1072'), np.str_('276'), np.str_('277'), np.str_('32766'), np.str_('768'), np.str_('783')]
A01E X: (288, 25, 1001)


In [27]:
# ============================
# BAND-PASS FILTER
# ============================

X_train_filtered = X_train.copy()
X_test_filtered = X_test.copy()

for i in range(X_train_filtered.shape[0]):
    X_train_filtered[i] = mne.filter.filter_data(
        X_train_filtered[i],
        sfreq=250,
        l_freq=8,
        h_freq=30,
        verbose=False
    )

for i in range(X_test_filtered.shape[0]):
    X_test_filtered[i] = mne.filter.filter_data(
        X_test_filtered[i],
        sfreq=250,
        l_freq=8,
        h_freq=30,
        verbose=False
    )

print("Training:", X_train_filtered.shape)
print("Testing:", X_test_filtered.shape)

Training: (288, 25, 1001)
Testing: (288, 25, 1001)


In [28]:
y_test = labels_data["classlabel"].reshape(-1)

print("Test labels:", y_test.shape)
print("Classes:", np.unique(y_test, return_counts=True))

Test labels: (288,)
Classes: (array([1, 2, 3, 4], dtype=uint8), array([72, 72, 72, 72]))


In [29]:
# ============================
# CSP
# ============================

csp = CSP(
    n_components=8,
    reg=None,
    log=True,
    norm_trace=False
)

X_train_csp = csp.fit_transform(
    X_train_filtered,
    y_train
)

X_test_csp = csp.transform(
    X_test_filtered
)

print("Training CSP:", X_train_csp.shape)
print("Testing CSP:", X_test_csp.shape)

Computing rank from data with rank=None
    Using tolerance 6.7e-05 (2.2e-16 eps * 25 dim * 1.2e+10  max singular value)
    Estimated rank (data): 25
    data: rank 25 computed from 25 data channels with 0 projectors
Reducing data rank from 25 -> 25
Estimating class=1 covariance using EMPIRICAL
Done.
Estimating class=2 covariance using EMPIRICAL
Done.
Estimating class=3 covariance using EMPIRICAL
Done.
Estimating class=4 covariance using EMPIRICAL
Done.
Training CSP: (288, 8)
Testing CSP: (288, 8)


In [30]:
# ============================
# RBF SVM
# ============================

svm = SVC(
    kernel="rbf",
    C=10,
    gamma="scale"
)

svm.fit(
    X_train_csp,
    y_train
)

print("SVM training completed.")

SVM training completed.


In [31]:
# ============================
# PREDICT A01E
# ============================

y_pred = svm.predict(X_test_csp)

print("Predictions:", y_pred.shape)
print("First 20 predictions:")
print(y_pred[:20])

Predictions: (288,)
First 20 predictions:
[4 3 3 1 3 1 2 4 3 4 1 4 2 4 4 3 3 4 4 1]


In [33]:
print("True labels:", np.unique(y_test, return_counts=True))
print("Predicted labels:", np.unique(y_pred, return_counts=True))

True labels: (array([1, 2, 3, 4], dtype=uint8), array([72, 72, 72, 72]))
Predicted labels: (array([1, 2, 3, 4]), array([97, 43, 55, 93]))


In [34]:
accuracy = accuracy_score(y_test, y_pred)

print("A01T → A01E Accuracy:", accuracy * 100, "%")

print("\nClassification Report:")

print(
    classification_report(
        y_test,
        y_pred,
        labels=[1, 2, 3, 4],
        target_names=[
            "Left Hand",
            "Right Hand",
            "Foot",
            "Tongue"
        ],
        zero_division=0
    )
)

A01T → A01E Accuracy: 67.70833333333334 %

Classification Report:
              precision    recall  f1-score   support

   Left Hand       0.66      0.89      0.76        72
  Right Hand       0.95      0.57      0.71        72
        Foot       0.58      0.44      0.50        72
      Tongue       0.62      0.81      0.70        72

    accuracy                           0.68       288
   macro avg       0.70      0.68      0.67       288
weighted avg       0.70      0.68      0.67       288

